# Confidence Classification MLP + t-SNE Mapping

이 노트북은 `metadata.csv`를 기준으로 `sound_id`별 audio/text `.npy` 임베딩을 로드하고, `class_idx`를 23차원 one-hot으로 변환한 뒤 `audio_emb + text_emb + class_onehot` feature로 confidence classifier를 학습합니다.

마지막에는 validation set의 마지막 hidden layer를 t-SNE로 2차원 시각화하고, 왼쪽은 true confidence, 오른쪽은 predicted confidence 기준으로 색칠합니다. 각 점이 어떤 데이터인지 확인할 수 있도록 t-SNE 좌표와 metadata를 CSV로 저장하고, Plotly가 설치되어 있으면 hover 가능한 interactive plot도 생성합니다.

In [2]:
%pip install torch scikit-learn

^C
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import importlib.util
import json
import random
import warnings

REQUIRED_PACKAGES = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'torch': 'torch',
    'sklearn': 'scikit-learn',
}
missing_packages = [pip_name for import_name, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(import_name) is None]
if missing_packages:
    raise ImportError(
        'Missing packages: ' + ', '.join(missing_packages) + '\n'
        '필요하면 노트북 셀에서 `%pip install ' + ' '.join(missing_packages) + '`를 먼저 실행하세요.'
    )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

DATA_DIR = ROOT / 'data'
METADATA_DIR = DATA_DIR / 'metadata'
FEATURE_DIR = DATA_DIR / 'features'
OUTPUT_DIR = ROOT / 'outputs' / 'confidence_mlp_tsne'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

ImportError: Missing packages: torch, scikit-learn
필요하면 노트북 셀에서 `%pip install torch scikit-learn`를 먼저 실행하세요.

## 1. Config

아래 경로만 현재 데이터 위치에 맞게 수정하면 됩니다. 기본값은 이 repo에서 기존 실험 코드가 사용하던 경로와 `metadata.csv` 후보를 함께 찾도록 되어 있습니다.

In [ ]:
CONFIG = {
    'seed': 42,
    'metadata_candidates': [
        ROOT / 'metadata.csv',
        METADATA_DIR / 'metadata.csv',
        METADATA_DIR / 'BSD10k_metadata.csv',
    ],
    'audio_dir': FEATURE_DIR / 'clap_audio_embeddings',
    'text_dir': FEATURE_DIR / 'clap_text_embeddings',
    'test_size': 0.2,
    'batch_size': 256,
    'epochs': 40,
    'patience': 7,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'hidden_dims': [512, 256],
    'dropout': 0.3,
    'num_workers': 0,
}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])

metadata_path = next((p for p in CONFIG['metadata_candidates'] if p.is_file()), None)
if metadata_path is None:
    print('Metadata candidates:')
    for p in CONFIG['metadata_candidates']:
        print(' -', p)
    raise FileNotFoundError('metadata.csv 또는 BSD10k_metadata.csv를 찾지 못했습니다. CONFIG의 metadata_candidates를 수정하세요.')

if not CONFIG['audio_dir'].is_dir():
    raise FileNotFoundError(f"audio embedding dir not found: {CONFIG['audio_dir']}")
if not CONFIG['text_dir'].is_dir():
    raise FileNotFoundError(f"text embedding dir not found: {CONFIG['text_dir']}")

print('metadata_path:', metadata_path)
print('audio_dir:', CONFIG['audio_dir'])
print('text_dir:', CONFIG['text_dir'])
print('output_dir:', OUTPUT_DIR)

## 2. Metadata 로드 및 정리

In [ ]:
df = pd.read_csv(metadata_path)
print(df.shape)
display(df.head())
print(df.columns.tolist())

In [ ]:
REQUIRED_COLUMNS = ['sound_id', 'class_idx', 'confidence']
missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

df = df.copy()
df['sound_id'] = df['sound_id'].astype(str).str.strip()
df['class_idx'] = df['class_idx'].astype(str).str.strip()

if 'class' in df.columns:
    df['class'] = df['class'].astype(str).str.strip()
if 'class_top' in df.columns:
    df['class_top'] = df['class_top'].astype(str).str.strip()

# 기존 confidence 실험과 동일하게 aggregate/invalid class code를 제거합니다.
invalid_class = (df['class_idx'].str.len() == 3) & (df['class_idx'].str.endswith(('99', '00')))
df = df.loc[~invalid_class].copy()

df['confidence'] = pd.to_numeric(df['confidence'], errors='coerce')
df = df[df['confidence'].isin([1, 2, 3, 4, 5])].copy()
df['confidence'] = df['confidence'].astype(int)
df = df.reset_index(drop=True)

print('clean shape:', df.shape)
print('confidence distribution')
display(df['confidence'].value_counts().sort_index().to_frame('count'))
print('class_idx unique:', df['class_idx'].nunique())
display(df['class_idx'].value_counts().sort_index().to_frame('count').head(30))

## 3. sound_id 기준 audio/text `.npy` 로드

embedding 파일이 둘 다 있는 row만 사용합니다.

In [ ]:
def load_embeddings_by_sound_id(df, audio_dir, text_dir):
    audio_rows, text_rows, kept_rows = [], [], []
    missing_audio, missing_text = [], []

    for row_idx, row in df.reset_index(drop=True).iterrows():
        sound_id = str(row['sound_id'])
        audio_path = Path(audio_dir) / f'{sound_id}.npy'
        text_path = Path(text_dir) / f'{sound_id}.npy'

        if not audio_path.is_file():
            missing_audio.append(sound_id)
            continue
        if not text_path.is_file():
            missing_text.append(sound_id)
            continue

        audio_rows.append(np.load(audio_path).reshape(-1).astype(np.float32))
        text_rows.append(np.load(text_path).reshape(-1).astype(np.float32))
        kept_rows.append(row_idx)

    if not kept_rows:
        raise RuntimeError(
            'audio/text embedding이 모두 존재하는 row가 없습니다. '
            f"audio_dir={audio_dir}, text_dir={text_dir} 경로를 확인하세요. "
            f"missing_audio 예시={missing_audio[:5]}, missing_text 예시={missing_text[:5]}"
        )

    kept_df = df.iloc[kept_rows].reset_index(drop=True)
    audio = np.vstack(audio_rows).astype(np.float32)
    text = np.vstack(text_rows).astype(np.float32)

    print(f'kept rows: {len(kept_rows):,} / {len(df):,}')
    print(f'missing audio: {len(missing_audio):,}, missing text: {len(missing_text):,}')
    print('audio shape:', audio.shape, 'text shape:', text.shape)
    return kept_df, audio, text, missing_audio, missing_text

df_model, audio_emb, text_emb, missing_audio, missing_text = load_embeddings_by_sound_id(
    df, CONFIG['audio_dir'], CONFIG['text_dir']
)

## 4. class_idx → 23차원 one-hot, feature concat

`class_idx` 값이 0-22가 아니라 101/102/... 같은 코드인 경우가 많으므로, metadata 안의 unique `class_idx`를 정렬한 뒤 0-22 column으로 매핑합니다.

In [ ]:
class_idx_values = sorted(df_model['class_idx'].astype(str).unique().tolist())
if len(class_idx_values) != 23:
    print(f'WARNING: class_idx unique count is {len(class_idx_values)}, not 23. 현재 unique 기준으로 one-hot을 만듭니다.')

class_to_col = {class_idx: i for i, class_idx in enumerate(class_idx_values)}
class_onehot = np.zeros((len(df_model), len(class_idx_values)), dtype=np.float32)
for row_i, class_idx in enumerate(df_model['class_idx'].astype(str)):
    class_onehot[row_i, class_to_col[class_idx]] = 1.0

X = np.concatenate([audio_emb, text_emb, class_onehot], axis=1).astype(np.float32)
y = (df_model['confidence'].to_numpy(dtype=np.int64) - 1)  # torch target: 0..4

print('feature shape:', X.shape)
print('audio dim:', audio_emb.shape[1], 'text dim:', text_emb.shape[1], 'class one-hot dim:', class_onehot.shape[1])
print('target shape:', y.shape, 'target labels:', sorted(np.unique(y + 1).tolist()))

with open(OUTPUT_DIR / 'class_idx_to_onehot_col.json', 'w', encoding='utf-8') as f:
    json.dump(class_to_col, f, indent=2, ensure_ascii=False)

display(pd.DataFrame({'class_idx': class_idx_values, 'onehot_col': range(len(class_idx_values))}).head(30))

## 5. Train/Valid split + Dataset/DataLoader

In [ ]:
idx = np.arange(len(df_model))
train_idx, valid_idx = train_test_split(
    idx,
    test_size=CONFIG['test_size'],
    random_state=CONFIG['seed'],
    stratify=y,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X[train_idx]).astype(np.float32)
X_valid = scaler.transform(X[valid_idx]).astype(np.float32)
y_train = y[train_idx].astype(np.int64)
y_valid = y[valid_idx].astype(np.int64)

df_train = df_model.iloc[train_idx].reset_index(drop=True)
df_valid = df_model.iloc[valid_idx].reset_index(drop=True)

class ConfidenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]

train_loader = DataLoader(
    ConfidenceDataset(X_train, y_train),
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
)
valid_loader = DataLoader(
    ConfidenceDataset(X_valid, y_valid),
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
)

print('train:', X_train.shape, 'valid:', X_valid.shape)
display(pd.DataFrame({'train': pd.Series(y_train + 1).value_counts().sort_index(), 'valid': pd.Series(y_valid + 1).value_counts().sort_index()}))

## 6. Confidence classification MLP 학습

In [ ]:
class ConfidenceMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(512, 256), n_classes=5, dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = hidden_dim
        self.hidden = nn.Sequential(*layers)
        self.classifier = nn.Linear(prev_dim, n_classes)

    def forward(self, x, return_hidden=False):
        h = self.hidden(x)
        logits = self.classifier(h)
        if return_hidden:
            return logits, h
        return logits

model = ConfidenceMLP(
    input_dim=X_train.shape[1],
    hidden_dims=CONFIG['hidden_dims'],
    n_classes=5,
    dropout=CONFIG['dropout'],
).to(DEVICE)

class_counts = np.bincount(y_train, minlength=5).astype(np.float32)
class_weights = class_counts.sum() / np.maximum(class_counts, 1.0)
class_weights = class_weights / class_weights.mean()
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])

print(model)
print('class weights:', class_weights)

In [ ]:
def predict_loader(model, loader):
    model.eval()
    losses, y_true, y_pred, probs, hidden_rows = [], [], [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits, h = model(xb, return_hidden=True)
            loss = criterion(logits, yb)
            p = torch.softmax(logits, dim=1)

            losses.append(loss.item() * len(yb))
            y_true.append(yb.cpu().numpy())
            y_pred.append(p.argmax(dim=1).cpu().numpy())
            probs.append(p.cpu().numpy())
            hidden_rows.append(h.cpu().numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    probs = np.vstack(probs)
    hidden = np.vstack(hidden_rows)
    return {
        'loss': np.sum(losses) / len(y_true),
        'y_true': y_true,
        'y_pred': y_pred,
        'probs': probs,
        'hidden': hidden,
    }

def metric_summary(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'precision_weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall_weighted': recall_score(y_true, y_pred, average='weighted', zero_division=0),
    }

best_state = None
best_valid_f1 = -1.0
bad_epochs = 0
history = []

for epoch in range(1, CONFIG['epochs'] + 1):
    model.train()
    train_loss = 0.0
    seen = 0
    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(yb)
        seen += len(yb)

    valid_out = predict_loader(model, valid_loader)
    valid_metrics = metric_summary(valid_out['y_true'], valid_out['y_pred'])
    row = {'epoch': epoch, 'train_loss': train_loss / seen, 'valid_loss': valid_out['loss'], **valid_metrics}
    history.append(row)

    if row['f1_macro'] > best_valid_f1:
        best_valid_f1 = row['f1_macro']
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1

    print(
        f"epoch {epoch:02d} | train_loss={row['train_loss']:.4f} | valid_loss={row['valid_loss']:.4f} | "
        f"acc={row['accuracy']:.4f} | f1_macro={row['f1_macro']:.4f} | "
        f"precision={row['precision_macro']:.4f} | recall={row['recall_macro']:.4f}"
    )

    if bad_epochs >= CONFIG['patience']:
        print('early stopping')
        break

model.load_state_dict(best_state)
torch.save(model.state_dict(), OUTPUT_DIR / 'confidence_mlp_best.pt')
history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / 'training_history.csv', index=False)
display(history_df.tail())

## 7. Accuracy, F1, Precision, Recall

In [ ]:
valid_out = predict_loader(model, valid_loader)
y_true = valid_out['y_true']
y_pred = valid_out['y_pred']
valid_probs = valid_out['probs']
valid_hidden = valid_out['hidden']

metrics = metric_summary(y_true, y_pred)
metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(OUTPUT_DIR / 'valid_metrics.csv', index=False)
display(metrics_df)

print(classification_report(y_true + 1, y_pred + 1, labels=[1, 2, 3, 4, 5], zero_division=0))

cm = confusion_matrix(y_true + 1, y_pred + 1, labels=[1, 2, 3, 4, 5])
cm_df = pd.DataFrame(cm, index=[f'true_{i}' for i in range(1, 6)], columns=[f'pred_{i}' for i in range(1, 6)])
display(cm_df)
cm_df.to_csv(OUTPUT_DIR / 'valid_confusion_matrix.csv')

## 8. Prediction distribution 확인

In [ ]:
dist_df = pd.DataFrame({
    'true_confidence': pd.Series(y_true + 1).value_counts().sort_index(),
    'predicted_confidence': pd.Series(y_pred + 1).value_counts().sort_index(),
}).fillna(0).astype(int)
display(dist_df)
dist_df.to_csv(OUTPUT_DIR / 'prediction_distribution.csv')

ax = dist_df.plot(kind='bar', figsize=(8, 4), rot=0)
ax.set_xlabel('confidence class')
ax.set_ylabel('count')
ax.set_title('True vs Predicted Confidence Distribution')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'prediction_distribution.png', dpi=160)
plt.show()

## 9. 마지막 hidden layer t-SNE 시각화

왼쪽 그림은 true confidence, 오른쪽 그림은 predicted confidence 기준입니다.

In [ ]:
n_valid = len(valid_hidden)
perplexity = min(30, max(5, (n_valid - 1) // 3))
print('valid hidden shape:', valid_hidden.shape, 't-SNE perplexity:', perplexity)

tsne = TSNE(
    n_components=2,
    perplexity=perplexity,
    init='pca',
    learning_rate='auto',
    random_state=CONFIG['seed'],
)
tsne_xy = tsne.fit_transform(valid_hidden)

mapping_cols = [c for c in ['sound_id', 'class_idx', 'class', 'class_top', 'title', 'tags', 'description'] if c in df_valid.columns]
mapping_df = df_valid[mapping_cols].copy()
mapping_df.insert(0, 'valid_row_id', np.arange(len(mapping_df)))
mapping_df['tsne_x'] = tsne_xy[:, 0]
mapping_df['tsne_y'] = tsne_xy[:, 1]
mapping_df['true_confidence'] = y_true + 1
mapping_df['predicted_confidence'] = y_pred + 1
mapping_df['predicted_confidence_score'] = (valid_probs * np.arange(1, 6)).sum(axis=1)
mapping_df['is_correct'] = mapping_df['true_confidence'] == mapping_df['predicted_confidence']
for c in range(5):
    mapping_df[f'prob_confidence_{c + 1}'] = valid_probs[:, c]

mapping_path = OUTPUT_DIR / 'valid_tsne_mapping.csv'
mapping_df.to_csv(mapping_path, index=False, encoding='utf-8-sig')
print('saved:', mapping_path)
display(mapping_df.head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
cmap = 'viridis'

sc0 = axes[0].scatter(mapping_df['tsne_x'], mapping_df['tsne_y'], c=mapping_df['true_confidence'], cmap=cmap, s=20, alpha=0.85)
axes[0].set_title('t-SNE by TRUE confidence')
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')

sc1 = axes[1].scatter(mapping_df['tsne_x'], mapping_df['tsne_y'], c=mapping_df['predicted_confidence'], cmap=cmap, s=20, alpha=0.85)
axes[1].set_title('t-SNE by PREDICTED confidence')
axes[1].set_xlabel('t-SNE 1')

for ax in axes:
    ax.grid(alpha=0.2)

cbar = fig.colorbar(sc1, ax=axes.ravel().tolist(), ticks=[1, 2, 3, 4, 5])
cbar.set_label('confidence')
plt.savefig(OUTPUT_DIR / 'hidden_tsne_true_vs_pred.png', dpi=180, bbox_inches='tight')
plt.show()

## 10. 각 점과 데이터 mapping 확인

`valid_tsne_mapping.csv`에 `valid_row_id`, `sound_id`, metadata, t-SNE 좌표, true/pred confidence, confidence probability를 저장했습니다. Plotly가 설치되어 있으면 hover로 각 점의 `sound_id`, `class_idx`, `title` 등을 바로 확인할 수 있습니다.

In [ ]:
def show_point(valid_row_id):
    row = mapping_df.loc[mapping_df['valid_row_id'] == valid_row_id]
    if row.empty:
        raise ValueError(f'valid_row_id={valid_row_id} not found')
    return row.T

# 예시: t-SNE plot에서 확인하고 싶은 valid_row_id를 넣으세요.
show_point(0)

In [ ]:
try:
    import plotly.express as px
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go

    hover_cols = [c for c in ['valid_row_id', 'sound_id', 'class_idx', 'class', 'class_top', 'title', 'true_confidence', 'predicted_confidence', 'predicted_confidence_score', 'is_correct'] if c in mapping_df.columns]

    fig_true = px.scatter(
        mapping_df,
        x='tsne_x',
        y='tsne_y',
        color='true_confidence',
        hover_data=hover_cols,
        color_continuous_scale='Viridis',
        title='TRUE confidence',
    )
    fig_pred = px.scatter(
        mapping_df,
        x='tsne_x',
        y='tsne_y',
        color='predicted_confidence',
        hover_data=hover_cols,
        color_continuous_scale='Viridis',
        title='PREDICTED confidence',
    )

    fig = make_subplots(rows=1, cols=2, subplot_titles=('TRUE confidence', 'PREDICTED confidence'))
    for trace in fig_true.data:
        fig.add_trace(trace, row=1, col=1)
    for trace in fig_pred.data:
        trace.showscale = False
        fig.add_trace(trace, row=1, col=2)
    fig.update_layout(width=1100, height=500, title='Hidden Layer t-SNE with Data Mapping')
    fig.write_html(OUTPUT_DIR / 'hidden_tsne_interactive.html')
    fig.show()
    print('saved:', OUTPUT_DIR / 'hidden_tsne_interactive.html')
except ImportError:
    print('plotly가 설치되어 있지 않아 interactive plot은 건너뜁니다. mapping_df 또는 valid_tsne_mapping.csv로 각 점을 확인하세요.')